# 🚦 TEST ĐẾM — YOLOv8 + supervision · 3 CHẾ ĐỘ · NGƯỜI & PHƯƠNG TIỆN

Hệ thống đếm giờ **chỉ dùng YOLOv8 + supervision** (bỏ LocateAnything). Test trên
**2 video** cho **2 bài toán**: 🚶 đếm NGƯỜI · 🚗 đếm PHƯƠNG TIỆN. Mỗi video đếm theo
**3 chế độ**:
- **TOÀN MÀN HÌNH** — đếm MỌI vật trong khung (đang có / đỉnh / tổng vật khác nhau).
- **ZONE** — đếm vật trong 1 VÙNG (polygon).
- **LINE** — đếm vật CẮT VẠCH (vào/ra).

**YOLO đã chỉnh để ÍT BỎ SÓT**: `yolov8x` (bản lớn nhất) · `imgsz=1280` (bắt vật nhỏ/xa) ·
`conf=0.15` (ngưỡng thấp) · `max_det=1000` (cảnh đông). Đổi nhanh:
`YOLO_WEIGHTS=yolov8m.pt` (nhanh hơn) · `YOLO_AUGMENT=1` (TTA, recall cao hơn nữa).

Mỗi nguồn xuất: **ẢNH** (YOLO bắt được gì) + **VIDEO** (đếm 3 chế độ cùng lúc). Cần **GPU T4**.

## 1) Cài đặt + tải code (YOLOv8 + supervision)

In [ ]:
import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print("📂 WORK =", WORK)
REPO = os.path.join(WORK, "VisionOS"); BR = "claude/locate-anything-test-suite-xwju2f"
if not os.path.isdir(os.path.join(REPO, ".git")):
    os.system(f"git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git {REPO}")
os.chdir(REPO)
os.system(f"git fetch -q origin {BR} && git checkout -q {BR} && git reset --hard -q origin/{BR}")
os.chdir(os.path.join(REPO, "VisionOS"))
print("📁 cwd =", os.getcwd(), "| nhánh =", BR)
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")
import torch
print("🖥️ GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "❌ CHƯA BẬT GPU! Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU")

## 2) Nạp YOLOv8 (đã FIX recall) + imports
`yolov8x` lần đầu tải ~130MB. Muốn nhanh hơn: đặt `os.environ['YOLO_WEIGHTS']='yolov8m.pt'`
TRƯỚC khi nạp. Muốn recall cao hơn nữa (chậm hơn): `os.environ['YOLO_AUGMENT']='1'`.

In [ ]:
import sys, time, math, subprocess
import numpy as np, cv2
import supervision as sv
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
from recognition.detectors import load_standard_detector
from recognition.sv_counting import _to_sv, _make_polygon_zone   # dùng lại helper repo

# (tuỳ chọn) os.environ["YOLO_WEIGHTS"]="yolov8m.pt"   # nhanh hơn
# (tuỳ chọn) os.environ["YOLO_AUGMENT"]="1"            # recall cao hơn (chậm)
det = load_standard_detector(backend="ultralytics")   # YOLOv8x, imgsz1280, conf0.15, max_det1000
det.load()
print("✅ YOLO sẵn sàng.")

## 3) Helper: đếm 3 CHẾ ĐỘ trong 1 lượt + ảnh nhận diện
Một lượt detect → cập nhật CẢ 3: toàn màn hình + zone + line; vẽ hết lên video.

In [ ]:
def new_tracker():
    for kw in (dict(track_activation_threshold=0.1, minimum_consecutive_frames=1, lost_track_buffer=120),
               dict(track_thresh=0.1), {}):
        try: return sv.ByteTrack(**kw)
        except TypeError: continue
    return sv.ByteTrack()

def _px(pt, w, h): return (int(pt[0] / 100 * w), int(pt[1] / 100 * h))

def detect_image(det, path, prompt, reso, n=4):
    """Chạy YOLO trên n frame → ảnh ghép cho thấy bắt được bao nhiêu (recall)."""
    w, h = reso
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    shots = []
    for f in np.linspace(total * 0.3, total * 0.7, n):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, fr = cap.read()
        if not ok: continue
        fr = cv2.resize(fr, reso)
        dr = det.detect(fr, prompt)
        for d in dr.detections:
            x1, y1, x2, y2 = (int(v) for v in d.bbox.as_xyxy())
            cv2.rectangle(fr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(fr, f"{prompt}: {len(dr.detections)} box", (8, 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        shots.append(fr)
    cap.release()
    return shots

def count_3modes(det, path, prompt, reso, line_pct, zone_pct, max_frames, out_mp4):
    """1 lượt detect → đếm TOÀN MÀN HÌNH + ZONE + LINE; vẽ + lưu video; trả 3 số."""
    w, h = reso
    tr = new_tracker()
    try: sm = sv.DetectionsSmoother(length=8)
    except Exception: sm = None
    (sx, sy), (ex, ey) = _px(line_pct[:2], w, h), _px(line_pct[2:], w, h)
    line = sv.LineZone(start=sv.Point(sx, sy), end=sv.Point(ex, ey))
    zpoly = np.array([_px(p, w, h) for p in zone_pct], dtype=np.int32)
    zone = _make_polygon_zone(sv, zpoly, w, h)
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), 12, (w, h))
    cap = cv2.VideoCapture(path); i = 0
    seen = set(); fs_peak = 0; z_peak = 0
    while i < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, reso)
        dr = det.detect(fr, prompt)
        sd = _to_sv(dr.detections, sv, np)
        sd = tr.update_with_detections(sd)
        if sm is not None:
            try: sd = sm.update_with_detections(sd)
            except Exception: pass
        if sd.tracker_id is not None:
            for t in sd.tracker_id:
                if t is not None: seen.add(int(t))
        line.trigger(sd)
        inz = np.asarray(zone.trigger(sd), dtype=bool) if len(sd) else np.zeros(0, bool)
        z_cur = int(inz.sum()); z_peak = max(z_peak, z_cur)
        fs_cur = int(len(sd)); fs_peak = max(fs_peak, fs_cur)
        # vẽ: box + id, vùng (xanh), vạch (vàng), banner 3 chế độ
        ids = list(sd.tracker_id) if sd.tracker_id is not None else [None] * len(sd)
        for b, t in zip(sd.xyxy.astype(int), ids):
            cv2.rectangle(fr, (b[0], b[1]), (b[2], b[3]), (0, 200, 0), 2)
            cv2.putText(fr, f"#{t}", (b[0], max(12, b[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
        ov = fr.copy(); cv2.fillPoly(ov, [zpoly], (0, 170, 0)); cv2.addWeighted(ov, 0.2, fr, 0.8, 0, fr)
        cv2.polylines(fr, [zpoly], True, (0, 200, 0), 2)
        cv2.line(fr, (sx, sy), (ex, ey), (0, 255, 255), 3)
        cv2.rectangle(fr, (0, 0), (w, 52), (0, 0, 0), -1)
        cv2.putText(fr, f"TOAN MH: dang {fs_cur} dinh {fs_peak} | tong (tracks) {len(seen)}",
                    (6, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(fr, f"ZONE: {z_cur} (dinh {z_peak})   LINE: in {line.in_count} out {line.out_count} "
                        f"total {line.in_count + line.out_count}",
                    (6, 44), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        vw.write(fr); i += 1
    vw.release(); cap.release()
    return dict(fullscreen=dict(peak=fs_peak, total=len(seen)),
                zone=dict(peak=z_peak),
                line=dict(in_=line.in_count, out=line.out_count, total=line.in_count + line.out_count),
                frames=i)
print("✅ Helper 3-chế-độ sẵn sàng.")

## 4) Cấu hình 2 nguồn + tải video
🚗 **PHƯƠNG TIỆN**: `vehicles-2.mp4` (giao lộ nhiều xe). 🚶 **NGƯỜI**: `market-square.mp4`
(quảng trường đông người, MÀU). Vạch/vùng theo % (0..100).

In [ ]:
def dl(name):
    p = os.path.join(WORK, name)
    if not os.path.exists(p):
        os.system(f'wget -q "https://media.roboflow.com/supervision/video-examples/{name}" -O "{p}"')
    return p if os.path.exists(p) and os.path.getsize(p) > 100000 else None

SOURCES = [
    dict(name="PHUONG TIEN", prompt="vehicle", path=dl("vehicles-2.mp4"),
         line=[3.5, 93.0, 93.0, 89.0],
         zone=[[15, 35], [85, 35], [85, 92], [15, 92]]),
    dict(name="NGUOI", prompt="person", path=dl("market-square.mp4"),
         line=[4.2, 94.4, 94.8, 92.1],
         zone=[[28, 32], [72, 32], [72, 85], [28, 85]]),
]
RESOLUTION = (1280, 720)   # xử lý ở độ phân giải cao → YOLO bắt vật nhỏ tốt (ít bỏ sót)
MAX_FRAMES = 150
for s in SOURCES:
    print(("OK  " if s["path"] else "MISSING  "), s["name"], "→", s["prompt"], "|", s["path"])

## 5) ▶️ CHẠY: mỗi nguồn → ẢNH nhận diện + VIDEO đếm 3 chế độ (LƯU HẾT)

In [ ]:
OUTDIR = os.path.join(WORK, "yolo_count_out"); os.makedirs(OUTDIR, exist_ok=True)
summary = []
for s in SOURCES:
    name, path = s["name"], s["path"]
    print(f"\n{'='*60}\n▶ {name}  ({s['prompt']})  {os.path.basename(str(path))}")
    if not path:
        print("   ❌ thiếu video (mạng?), bỏ qua"); continue
    # (1) ẢNH nhận diện — xem YOLO bắt được bao nhiêu (recall)
    shots = detect_image(det, path, s["prompt"], RESOLUTION, n=4)
    if shots:
        cols = len(shots); fig, ax = plt.subplots(1, cols, figsize=(5 * cols, 3))
        for a, im in zip(np.atleast_1d(ax), shots):
            a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.axis("off")
        plt.suptitle(f"{name} - YOLO nhan dien '{s['prompt']}'", fontsize=11)
        gpng = os.path.join(OUTDIR, f"{name}_detect.png".replace(" ", "_"))
        plt.tight_layout(); plt.savefig(gpng, dpi=110, bbox_inches="tight"); plt.show()
    else:
        gpng = None
    # (2) VIDEO đếm 3 chế độ
    mp4 = os.path.join(OUTDIR, f"{name}_count3.mp4".replace(" ", "_"))
    print(f"   Đếm 3 chế độ ({MAX_FRAMES} frame)…")
    r = count_3modes(det, path, s["prompt"], RESOLUTION, s["line"], s["zone"], MAX_FRAMES, mp4)
    print(f"   → TOÀN MH: đỉnh {r['fullscreen']['peak']}, tổng {r['fullscreen']['total']}")
    print(f"     ZONE   : đỉnh {r['zone']['peak']}")
    print(f"     LINE   : in {r['line']['in_']} / out {r['line']['out']} / total {r['line']['total']}")
    summary.append(dict(name=name, detect=gpng, mp4=mp4, r=r))

print("\n" + "=" * 60 + "\nTỔNG HỢP — 3 chế độ đếm\n" + "=" * 60)
print(f"{'Nguồn':14}{'TOÀN MH (đỉnh/tổng)':22}{'ZONE (đỉnh)':14}{'LINE (total)':12}")
for s in summary:
    r = s["r"]
    fs = f"{r['fullscreen']['peak']}/{r['fullscreen']['total']}"
    print(f"{s['name']:14}{fs:22}{r['zone']['peak']:<14}{r['line']['total']:<12}")

## 6) 🎥 Xem/tải ĐẦY ĐỦ ảnh + video từng nguồn (H.264)

In [ ]:
from IPython.display import Video, display, Markdown, Image
for s in summary:
    r = s["r"]
    display(Markdown(f"## {s['name']} — TOÀN MH đỉnh {r['fullscreen']['peak']}/tổng {r['fullscreen']['total']} · "
                     f"ZONE đỉnh {r['zone']['peak']} · LINE total {r['line']['total']}"))
    if s["detect"] and os.path.exists(s["detect"]):
        display(Image(filename=s["detect"], width=820))
    mp4 = s["mp4"]; h264 = mp4.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp4,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    display(Video(h264 if os.path.exists(h264) else mp4, embed=True, width=680))
print("\n💾 Ảnh + video đã lưu ở:", OUTDIR, "(Kaggle: tải ở panel Output).")

### Đọc kết quả — 3 chế độ đếm
- **TOÀN MÀN HÌNH**: *đang* = số vật trong khung lúc đó · *đỉnh* = lúc đông nhất ·
  *tổng (tracks)* = số vật KHÁC NHAU đã thấy suốt video (≈ "có bao nhiêu người/xe").
- **ZONE**: số vật đang trong VÙNG (đỉnh = đông nhất trong vùng). Hợp đếm mật độ 1 khu vực.
- **LINE**: số vật CẮT VẠCH (in/out/total). Hợp đếm vào/ra 1 lối.

**YOLO bỏ sót?** Đã chỉnh recall cao (yolov8x/imgsz1280/conf0.15). Vẫn sót vật rất nhỏ/khuất:
bật `os.environ['YOLO_AUGMENT']='1'` (Cell 2, chậm hơn) hoặc tăng `RESOLUTION`. **Đếm LINE=0?**
do vạch lệch dòng đi — sửa `line` trong Cell 4 (toạ độ %). **Ảnh trắng đen** không sao với YOLO:
YOLO nhận diện người/xe KHÔNG dựa vào màu (màu chỉ cần cho mô tả — đã bỏ ở bản này).